### Env setup

#### Goal: Prepare your local system for unstructured text processing.

In [1]:
pip install langchain langchain-community langchain-chroma langchain-ollama xmltodict pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Data Ingestion (PubMed API)

#### Goal: Retrieve live medical literature abstracts directly into LangChain.

###### Instead of loading a static CSV, we will use LangChain's built-in PubMedLoader to fetch recent biomedical abstracts based on targeted drug-event queries.

In [4]:
from langchain_community.document_loaders.pubmed import PubMedLoader

# Define your targeted literature search (e.g., a specific checkpoint inhibitor)
query = "pembrolizumab AND hepatotoxicity"

# Fetch the most recent abstracts matching the query
print("Fetching abstracts from PubMed...")
loader = PubMedLoader(query=query, load_max_docs=50)
docs = loader.load()

print(f"✅ Successfully loaded {len(docs)} literature abstracts.")

Fetching abstracts from PubMed...
✅ Successfully loaded 50 literature abstracts.


### Step 3: Embeddings & Vector Store

#### Goal: Convert the unstructured abstracts into embeddings and store them in ChromaDB.

##### Because we are dealing with paragraphs of text (abstracts) rather than flattened database rows, semantic embeddings will perfectly capture the clinical context of the literature.

In [5]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

# Initialize local embedding model
embeddings = OllamaEmbeddings(model="mxbai-embed-large")
db_location = "./chroma_mlm_db"

# Create and persist the vector store
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=db_location,
    collection_name="mlm_abstracts"
)

print("✅ Abstracts embedded and stored in ChromaDB.")

✅ Abstracts embedded and stored in ChromaDB.


### The Triage Prompt & LLM Chain

#### Goal: Instruct the LLM to act as a pharmacovigilance literature reviewer to determine if an abstract contains a valid ICSR.

##### A valid ICSR requires specific elements: an identifiable patient, a suspect drug, an adverse event, and an identifiable reporter. We will strictly prompt the LLM to look for these elements.

In [6]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

llm = OllamaLLM(model="llama3.2", temperature=0.1)

template = """
You are an expert pharmacovigilance literature reviewer.
Review the following medical abstract and extract the required information to determine if it represents a valid Individual Case Safety Report (ICSR).

Abstract: {abstract}

Provide a structured assessment containing exactly these fields:
1. Patient Details: [Extract age, sex, or state 'Not specified']
2. Suspect Drug: [Extract the drug name]
3. Adverse Event: [Extract the adverse event]
4. Is Valid ICSR: [Yes/No - Answer Yes ONLY if patient, drug, and event are all explicitly mentioned as a case study]
5. Rationale: [One short sentence explaining why]
"""

prompt = ChatPromptTemplate.from_template(template)
triage_chain = prompt | llm

### Retrieval and Agent Workflow

#### Goal: Query the database for relevant literature and process each abstract through the triage chain.

In [7]:
import pandas as pd
import datetime

# Retrieve abstracts mentioning specific severity or case report indicators
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

search_query = "case report of severe hepatic injury"
retrieved_abstracts = retriever.invoke(search_query)

audit_log = []

for doc in retrieved_abstracts:
    # Run the LLM triage on the abstract text
    triage_result = triage_chain.invoke({"abstract": doc.page_content})
    
    # Log the metadata (PubMed ID, Title) and the LLM's decision
    audit_log.append({
        "timestamp": datetime.datetime.now().isoformat(),
        "pubmed_id": doc.metadata.get("uid"),
        "title": doc.metadata.get("Title"),
        "triage_assessment": triage_result.strip()
    })

# Convert to DataFrame for regulatory tracking
audit_df = pd.DataFrame(audit_log)
audit_df.to_csv("mlm_triage_audit_log.csv", index=False)
print("✅ Triage complete. Audit log saved.")

✅ Triage complete. Audit log saved.
